# High-confidence accuracy policy

## tl;dr

The core model's full-match holdout accuracy is 61.93%. A validation-selected
confidence threshold of 0.50 produces **73.19% holdout accuracy** on **65.60%
coverage**. Uncertain matches are marked `abstain`; their probabilities remain
available for simulation and probabilistic scoring.

## Context & Methods

The threshold is selected on rolling-origin predictions only. It must achieve
at least 65% accuracy separately in every validation year, with at least 100
selected predictions per year. The fixed 2025-2026 holdout is used only for the
final audit.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

results = pd.read_csv(ROOT / "reports/selective_accuracy/results.csv")
selection = json.loads(
    (ROOT / "reports/selective_accuracy/selection.json").read_text()
)
predictions = pd.read_csv(
    ROOT / "reports/selective_accuracy/holdout_predictions.csv"
)

## Results

In [2]:
results[[
    "split", "year", "threshold", "coverage", "selective_accuracy",
    "full_accuracy", "selected_matches", "total_matches"
]]

,split,year,threshold,coverage,selective_accuracy,full_accuracy,selected_matches,total_matches
0,pooled_rolling_origin,NaN,0.5,0.660185,0.695453,0.606759,3497,5297
1,rolling_year,2018.0,0.5,0.590958,0.681239,0.574812,549,929
2,rolling_year,2021.0,0.5,0.689686,0.754226,0.653812,769,1115
3,rolling_year,2022.0,0.5,0.588660,0.684764,0.582474,571,970
4,rolling_year,2023.0,0.5,0.694497,0.706284,0.625237,732,1054
5,rolling_year,2024.0,0.5,0.712775,0.650685,0.591538,876,1229
6,holdout_2025_2026,NaN,0.5,0.655963,0.731935,0.619266,858,1308


In [3]:
summary = pd.DataFrame([
    {
        "metric": "Rolling selected accuracy",
        "value": selection["validation"]["selective_accuracy"],
    },
    {
        "metric": "Rolling coverage",
        "value": selection["validation"]["coverage"],
    },
    {
        "metric": "Holdout selected accuracy",
        "value": selection["holdout"]["selective_accuracy"],
    },
    {
        "metric": "Holdout coverage",
        "value": selection["holdout"]["coverage"],
    },
    {
        "metric": "Holdout full accuracy",
        "value": selection["holdout"]["full_accuracy"],
    },
])
summary

,metric,value
0,Rolling selected accuracy,0.695453
1,Rolling coverage,0.660185
2,Holdout selected accuracy,0.731935
3,Holdout coverage,0.655963
4,Holdout full accuracy,0.619266


## Prediction output

In [4]:
predictions[[
    "team1", "team2", "p_team1_win", "p_draw", "p_team2_win",
    "predicted_label", "confidence", "is_high_confidence"
]].head(20)

,team1,team2,p_team1_win,p_draw,p_team2_win,predicted_label,confidence,is_high_confidence
0,Vietnam,Thailand,0.377773,0.279515,0.342712,abstain,0.377773,False
1,Zanzibar,Tanzania,0.354150,0.295882,0.349968,abstain,0.354150,False
2,Burkina Faso,Kenya,0.534471,0.312948,0.152580,team1_win,0.534471,True
3,Oman,Bahrain,0.390811,0.343179,0.266011,abstain,0.390811,False
4,Thailand,Vietnam,0.550600,0.262102,0.187298,team1_win,0.550600,True
5,Zanzibar,Burkina Faso,0.395953,0.267375,0.336672,abstain,0.395953,False
6,Tanzania,Kenya,0.354782,0.376492,0.268726,abstain,0.376492,False
7,Tanzania,Burkina Faso,0.189438,0.363553,0.447009,abstain,0.447009,False
8,Zanzibar,Kenya,0.450607,0.297136,0.252258,abstain,0.450607,False
9,Zanzibar,Burkina Faso,0.346531,0.295436,0.358033,abstain,0.358033,False


## Takeaways

1. The requested 65% target is exceeded for the operational high-confidence
   subset, including every rolling validation year.
2. Coverage is part of the contract: approximately one third of matches are
   deliberately not assigned a hard pick.
3. Probability metrics and tournament simulation remain based on all matches.
4. Reporting selected accuracy without coverage would be misleading, so both
   are emitted in every artifact.